In [344]:
def numerical_gradient(f, var, h=1e-5):
    """Central difference: f'(x) ≈ (f(x+h) - f(x-h)) / (2h)"""
    x = var.value

    var.value = x + h
    fwd = f(var).value

    var.value = x - h
    bwd = f(var).value

    var.value = x  # Restore

    return (fwd - bwd) / (2 * h)

In [345]:
class Scalar:
    def __init__(self, value, parents=(), op=""):
        self.value = float(value)
        self.grad = 0.0
        self.op = op
        self.parents = parents
        self._backward = lambda: None

    def __repr__(self):
        return f"Scalar(value={self.value:.4f}, grad={self.grad:.4f})"

    # ----- addition -----
    def __add__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value + other.value, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    # ----- multiplication -----
    def __mul__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value * other.value, (self, other), "*")
        def _backward():
            self.grad += other.value * out.grad
            other.grad += self.value * out.grad
        out._backward = _backward
        return out

    # ----- power (constant exponent) -----
    def __pow__(self, exponent):
        out = Scalar(self.value ** exponent, (self,), f"**{exponent}")
        def _backward():
            self.grad += exponent * (self.value ** (exponent - 1)) * out.grad
        out._backward = _backward
        return out

    # ----- backward engine (topological sort) -----
    def backward(self):
        topo = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for p in v.parents:
                    build(p)
                topo.append(v)
        build(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

In [346]:
a = Scalar(2.0)
b = Scalar(3.0)
c = Scalar(4.0)

L = (a * b + c) ** 2

L.backward()

print("Gradients from backprop:")
print(f"  dL/da = {a.grad:.6f}")
print(f"  dL/db = {b.grad:.6f}")
print(f"  dL/dc = {c.grad:.6f}")

Gradients from backprop:
  dL/da = 60.000000
  dL/db = 40.000000
  dL/dc = 20.000000


In [347]:
# Define functions that vary one variable at a time
def f_a(x):
    return (x * b + c) ** 2

def f_b(x):
    return (a * x + c) ** 2

def f_c(x):
    return (a * b + x) ** 2

num_a = numerical_gradient(f_a, a)
num_b = numerical_gradient(f_b, b)
num_c = numerical_gradient(f_c, c)

print("Numerical gradients:")
print(f"  dL/da ≈ {num_a:.6f}")
print(f"  dL/db ≈ {num_b:.6f}")
print(f"  dL/dc ≈ {num_c:.6f}")

# Verify
tol = 1e-5
assert abs(a.grad - num_a) < tol, "Gradient w.r.t a failed"
assert abs(b.grad - num_b) < tol, "Gradient w.r.t b failed"
assert abs(c.grad - num_c) < tol, "Gradient w.r.t c failed"

print("\n✅ All gradients match numerical approximation.")

Numerical gradients:
  dL/da ≈ 60.000000
  dL/db ≈ 40.000000
  dL/dc ≈ 20.000000

✅ All gradients match numerical approximation.
